# CosmUFR Run 4 — corner-plot demo

This notebook builds a standard 8x8 cosmology corner plot from a *single*
CosmUFR forward pass. The contours are **analytic Fisher-matrix ellipses**
computed from the per-parameter uncertainties the model already returns —
no MCMC chains, no extra model outputs.

Because the released model returns 8 per-parameter variances and not a full
covariance matrix, the Fisher matrix is taken to be **diagonal**, so the
off-diagonal 2D ellipses are axis-aligned. This is a deliberate honest
limitation — see the discussion at the end of the notebook.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import cosmufr

ckpt_local = Path("../_local_ckpt/best.pt")
model = cosmufr.load_model(ckpt_path=str(ckpt_local) if ckpt_local.exists() else None)

arr = np.load("synthetic_pk.npy")
result = cosmufr.infer(arr[0], arr[1], model=model)

labels = cosmufr.PARAM_LABELS    # ['Om','s8','h','ns','Ob','w0','mv','wa']
mu     = result.params_array     # shape (8,)
sig    = result.sigmas_array     # shape (8,)
for lbl, m, s in zip(labels, mu, sig):
    print(f"  {lbl:>3s}  =  {m:8.4f}  +/-  {s:.4f}")

In [ ]:
# Helper: 1-sigma and 2-sigma ellipse on axes (i, j) of a diagonal Fisher
# covariance with marginal sigmas (sig_i, sig_j). Axis-aligned because we have
# no off-diagonal information.

# Chi-square confidence levels for 2D Gaussian (k=2):
#   1-sigma -> deltachi2 = 2.30,  2-sigma -> 6.18
CONF_1S, CONF_2S = 2.30, 6.18

def ellipse_axes(sig_x, sig_y, conf):
    """Return (width, height) for a Fisher ellipse at confidence `conf`."""
    return 2 * np.sqrt(conf) * sig_x, 2 * np.sqrt(conf) * sig_y

n = len(labels)
fig, axes = plt.subplots(n, n, figsize=(14, 14))

# Plot range: +/- 3 sigma around the central estimate per axis
lo = mu - 3 * sig
hi = mu + 3 * sig

for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        if j > i:
            ax.set_visible(False)
            continue

        if i == j:
            # 1D marginal: Gaussian PDF
            x = np.linspace(lo[i], hi[i], 200)
            pdf = np.exp(-0.5 * ((x - mu[i]) / sig[i]) ** 2)
            ax.plot(x, pdf, color="C0")
            ax.axvline(mu[i], color="k", lw=0.7, ls="--")
            ax.set_xlim(lo[i], hi[i])
            ax.set_yticks([])
        else:
            # 2D marginal: 1- and 2-sigma Fisher ellipses
            for conf, color, alpha in [(CONF_2S, "C0", 0.18), (CONF_1S, "C0", 0.45)]:
                w, h = ellipse_axes(sig[j], sig[i], conf)
                ell = Ellipse((mu[j], mu[i]), w, h,
                              facecolor=color, edgecolor=color, alpha=alpha, lw=1.0)
                ax.add_patch(ell)
            ax.scatter([mu[j]], [mu[i]], s=8, color="k", zorder=5)
            ax.set_xlim(lo[j], hi[j])
            ax.set_ylim(lo[i], hi[i])

        # Axis labels only on the outermost row / column
        if i == n - 1:
            ax.set_xlabel(labels[j])
        else:
            ax.set_xticklabels([])
        if j == 0 and i > 0:
            ax.set_ylabel(labels[i])
        elif i == 0 and j == 0:
            pass  # diagonal, no y-label needed
        else:
            ax.set_yticklabels([])

        ax.tick_params(axis="both", labelsize=8)

fig.suptitle("CosmUFR Run 4 — Fisher corner plot from per-parameter sigmas",
             fontsize=14, y=0.92)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("corner_plot.png", dpi=120, bbox_inches="tight")
print("Saved corner_plot.png")

## Honest caveats

* The released model returns **per-parameter variances**, not a full
  posterior covariance. The off-diagonal panels above therefore use a
  diagonal Fisher matrix and the ellipses are axis-aligned. A real MCMC
  posterior would generally be tilted.
* Several parameters (h, ns, Ob, w0, mv, wa) are still YELLOW on this
  Run 4 release — the central values are biased and the sigmas are
  under-calibrated (overall ECE = 0.39). Read the contours as a sketch of
  the model's confidence, not a science-quality posterior.
* For an MCMC-quality corner plot, a full covariance head plus posterior
  calibration is on the planned Run 8 roadmap.